In [0]:
# =============================================================
# NOTEBOOK:  06_bronze_api_currency
# PURPOSE:   Ingest current data from api into Bronze Delta
# SOURCE:    currency api
# TARGET:    Bronze Delta Tables (bronze/tables/external_api/currency_exchange_rates/)
# PROJECT:   RetailMax Lakehouse
# =============================================================

import json
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import uuid
from delta.tables import DeltaTable
import requests


# ── Load Config ───────────────────────────────────────────────
config_path = "/Workspace/Repos/retailmax-lakehouse/retailmax-lakehouse/configs/dev/config.json"

with open(config_path, 'r') as f:
    config = json.load(f)

# ── Storage Paths ─────────────────────────────────────────────
storage_account = config['storage']['account_name']
bronze_path     = config['storage']['bronze_path']
scope_name      = config['secret_scope']

# ── Retrieve Secrets ──────────────────────────────────────────
client_id     = dbutils.secrets.get(scope=scope_name, key="sp-client-id")
client_secret = dbutils.secrets.get(scope=scope_name, key="sp-client-secret")
tenant_id     = dbutils.secrets.get(scope=scope_name, key="sp-tenant-id")

# ── Configure Spark for ADLS ──────────────────────────────────
spark.conf.set(
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net",
    "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net",
    client_id
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",
    client_secret
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)



In [0]:
api_endpoint = 'https://open.er-api.com/v6/latest/USD'

response = requests.get(api_endpoint,timeout=30)
if response.status_code == 200:
    data = response.json()
    currency_data = [{'api_endpoint':api_endpoint,'base_currency':data["base_code"],'raw_response':json.dumps(data)}]
    # print(currency_data)

    df = spark.createDataFrame(currency_data)
    # display(df)

    batch_id = uuid.uuid4().hex
    df = (df
        .withColumn("time_last_update_utc", F.lit(data.get("time_last_update_utc")))
        .withColumn("_load_timestamp", F.current_timestamp())
        .withColumn("_load_date", F.current_date())
        .withColumn("_batch_id", F.lit(batch_id))
    )
    # display(df)

    df.write\
        .mode('append')\
        .format('delta')\
        .save(bronze_path + "tables/external_api/currency_exchange_rates/")

else:
    raise Exception(f"API call failed: {response.status_code} - {response.text}")
    



In [0]:
df = spark.read.format("delta")\
            .load(bronze_path + "tables/external_api/currency_exchange_rates/")
df.show(truncate=False)